# Testing the TSFM model catalog tools

A hands-on walkthrough of the **7 model-catalog write tools**. Run the cells top to bottom and
read each response: this notebook **does not assert pass/fail**, it shows you what each tool
returns so you can compare against what you expect.

| # | tool | what it does |
|---|------|--------------|
| 13 | `model_template` | the card shape to author against |
| 12 | `register_model` | add a card to the catalog |
| 18 | `resolve_model` | preflight that a card can load |
| 15 | `update_model` | patch fields on a card |
| 14 | `register_finetuned` | add a fine-tune card, with lineage |
| 17 | `new_model_version` | supersede a card |
| 16 | `deprecate_model` | retire a card |

Calls go through **MCPHub / ToolUniverse**, the same path an agent uses:

```
ToolUniverse --> stdio --> tsfm-mcp-server --> CouchStore --> CouchDB
```

Nothing is imported from `servers.tsfm` - the server runs as a separate process.

## 0. Prerequisites

From the repo root, **before** starting Jupyter:

```bash
uv run python -m ipykernel install --user \
    --name assetopsbench-mcp --display-name "assetopsbench-mcp (uv)"

docker compose -f src/couchdb/docker-compose.yaml up -d   # CouchDB on :5984

python3 src/couchdb/init_data.py --reset                  # load the catalogs

uv run jupyter lab test_model_catalog_tools.ipynb
```

`--reset` **drops** the databases first. Only the `default` scenario carries the TSFM
catalogs - `scenario_1` / `scenario_2` hold work orders only.

You can watch the data in CouchDB's web UI at <http://localhost:5984/_utils> (admin/password).

In [1]:
import json, os, sys

# Point at the repo root: adjust if this notebook is not in the repo root.
REPO = os.path.abspath(os.environ.get('AOB_REPO', '.'))
SRC  = os.path.join(REPO, 'src')
sys.path.insert(0, SRC)

# The tsfm server is spawned as a SUBPROCESS and inherits this env, so PYTHONPATH must
# be set here - sys.path only affects this notebook, not the child process.
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('COUCHDB_URL', 'http://localhost:5984')
os.environ.setdefault('COUCHDB_USERNAME', 'admin')
os.environ.setdefault('COUCHDB_PASSWORD', 'password')
os.environ['TSFM_STORE'] = 'couch'      # 'memory' to run without CouchDB

print('repo      :', REPO)
print('couchdb   :', os.environ['COUCHDB_URL'])
print('store     :', os.environ['TSFM_STORE'])

repo      : /Users/dhaval/Documents/GitHub/AssetOpsBench/notebook
couchdb   : http://localhost:5984
store     : couch


### Is CouchDB up and seeded?

Expect `model_catalog` to exist. On a fresh `--reset` it holds **1** card (`ttm_96_28`).

In [2]:
import urllib.request, base64

def couch(path):
    url = os.environ['COUCHDB_URL'].rstrip('/') + path
    req = urllib.request.Request(url)
    tok = base64.b64encode(f"{os.environ['COUCHDB_USERNAME']}:{os.environ['COUCHDB_PASSWORD']}".encode()).decode()
    req.add_header('Authorization', 'Basic ' + tok)
    return json.load(urllib.request.urlopen(req, timeout=10))

try:
    print('databases    :', couch('/_all_dbs'))
    print('model_catalog:', couch('/model_catalog')['doc_count'], 'docs')
except Exception as e:
    print('CouchDB not reachable ->', e)
    print('Start it, or set TSFM_STORE=memory in the cell above.')

databases    : ['_replicator', '_users', 'asset', 'catalog', 'failure_code', 'failure_mode', 'feature_catalog', 'iot', 'model_catalog', 'vibration', 'workorder']
model_catalog: 11 docs


## 1. Connect through MCPHub

`load_tools` spawns the tsfm server as a subprocess and discovers its tools over stdio.

In [3]:
from mcphub import ToolUniverse

# 'uv run tsfm-mcp-server' is the normal launcher; this fallback needs no uv.
SERVER_CMD = os.environ.get('SERVER_CMD', f'{sys.executable} -m servers.tsfm.main').split()

tu = ToolUniverse(servers={'tsfm': SERVER_CMD})
n = tu.load_tools(servers=['tsfm'])
print(f'{n} tools discovered\n')
print('\n'.join(sorted(tu.all_tools)))

20 tools discovered

tsfm.characterize_series
tsfm.count_models
tsfm.data_quality
tsfm.deprecate_model
tsfm.describe_candidates
tsfm.describe_models
tsfm.find_models
tsfm.get_model_lineage
tsfm.hf_stats
tsfm.list_domains
tsfm.list_models
tsfm.list_tasks
tsfm.model_template
tsfm.new_model_version
tsfm.profile_series
tsfm.register_finetuned
tsfm.register_model
tsfm.resolve_model
tsfm.search_models
tsfm.update_model


## 13. `model_template` - what shape is a card?

Start here. It reads nothing from the database; it just tells you the contract.

**Expect:** `required_fields = [model_id, description, task_ids]`, the 5 `pointer_choices`
(the ways a card can reference a model), and a filled `example` that registers as-is.

### A note on the response shape

FastMCP is **not uniform** about this, and you will see it below:

* a tool annotated `-> Union[X, ErrorResult]` comes back **wrapped**: `{"result": {...}}`
* `model_template`, annotated `-> ModelTemplateResult`, comes back **flat**

The raw response is printed as-is so the difference is visible.

In [4]:
r = tu.run({
    'name': 'tsfm.model_template',
    'arguments': {},
})
print(json.dumps(r, indent=2, default=str))

{
  "required_fields": [
    "model_id",
    "description",
    "task_ids"
  ],
  "pointer_choices": [
    "sktime_class (+ params)  - resolve & construct via sktime, e.g. Est(**params)",
    "hf_repo                  - load weights lazily from HuggingFace",
    "artifact_path            - local checkpoint directory",
    "remote_endpoint          - hosted inference service",
    "model_checkpoint         - toolkit checkpoint (e.g. anomalykits://...)"
  ],
  "optional_fields": [
    "model_family",
    "domain",
    "context_length",
    "prediction_length",
    "provenance",
    "base_model_id",
    "usage_modes",
    "param_hints",
    "training_regime",
    "frequency",
    "tags"
  ],
  "resolution_rules": [
    "a card must be resolvable via at least one pointer_choice (else it is a catalog-only stub)",
    "provenance='finetuned' requires base_model_id (lineage)",
    "context_length / prediction_length must be >= 0"
  ],
  "example": {
    "model_id": "chronos_t5_small",
    "de

## 12. `register_model` - add a real model

We register [`ibm-granite/granite-timeseries-ttm-r1`](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r1):
**805,280 parameters** - the first sub-1M "tiny" time-series foundation model.

A card is a **pointer**: `sktime_class` + `params.model_path` say how to build and load it.
No weights are stored here.

**Expect:** `status = registered`, `id = hub_ttm_r1`.

In [5]:
R1  = 'ibm-granite/granite-timeseries-ttm-r1'
R2  = 'ibm-granite/granite-timeseries-ttm-r2'
TTM = 'sktime.forecasting.ttm.TinyTimeMixerForecaster'

r = tu.run({
    'name': 'tsfm.register_model',
    'arguments': {'model': {
        'model_id': 'hub_ttm_r1',
        'description': 'IBM Granite TinyTimeMixer R1: 805K-param tiny TS foundation forecaster.',
        'task_ids': ['tsfm_forecasting'],
        'sktime_class': TTM,
        'params': {'model_path': R1},
        'hf_repo': R1,
        'model_family': 'TinyTimeMixer',
        'context_length': 512,
        'prediction_length': 96,
        'tags': ['foundation', 'tiny'],
        'version': '1',
    }},
})
print(json.dumps(r, indent=2, default=str))

{
  "result": {
    "error": "model 'hub_ttm_r1' already exists; use new_model_version to supersede it, or update_model to patch it"
  }
}


### Try a bad card

**Expect:** an error naming both problems - `description` under 3 chars, `task_ids` empty.

In [6]:
r = tu.run({
    'name': 'tsfm.register_model',
    'arguments': {'model': {'model_id': 'bad', 'description': 'ab', 'task_ids': []}},
})
print(json.dumps(r, indent=2, default=str))

{
  "result": {
    "error": "2 validation errors for ModelCard\ndescription\n  String should have at least 3 characters [type=string_too_short, input_value='ab', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.12/v/string_too_short\ntask_ids\n  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list]\n    For further information visit https://errors.pydantic.dev/2.12/v/too_short"
  }
}


## 18. `resolve_model` - can it actually load?

A read-only preflight. It checks the `sktime_class` is importable and reports where the
weights come from. It does **not** download anything.

**Expect:** `resolvable = true`, `weights_from` = the HF repo, `training_regime = zero_shot`.

In [7]:
r = tu.run({
    'name': 'tsfm.resolve_model',
    'arguments': {'model_id': 'hub_ttm_r1'},
})
print(json.dumps(r, indent=2, default=str))

{
  "result": {
    "model_id": "hub_ttm_r1",
    "resolvable": true,
    "reason": "importable sktime_class + params; weights load lazily at fit",
    "sktime_class": "sktime.forecasting.ttm.TinyTimeMixerForecaster",
    "training_regime": "zero_shot",
    "weights_from": "ibm-granite/granite-timeseries-ttm-r1"
  }
}


**Expect:** a clear `not found` error.

In [8]:
r = tu.run({
    'name': 'tsfm.resolve_model',
    'arguments': {'model_id': 'no_such_model'},
})
print(json.dumps(r, indent=2, default=str))

{
  "result": {
    "error": "model 'no_such_model' not found"
  }
}


## 15. `update_model` - patch a card

Merges `fields`, stamps `updated_at`, and re-validates. An invalid patch is rejected.

**Expect:** `domain = energy` and a fresh `updated_at`. Note the card comes back **flat**
here, whereas `register_model` nested it under `card`.

In [9]:
r = tu.run({
    'name': 'tsfm.update_model',
    'arguments': {'model_id': 'hub_ttm_r1', 'fields': {'domain': 'finance'}},
})
print(json.dumps(r, indent=2, default=str))

{
  "result": {
    "model_id": "hub_ttm_r1",
    "description": "IBM Granite TinyTimeMixer R1: 805K-param tiny TS foundation forecaster.",
    "task_ids": [
      "tsfm_forecasting"
    ],
    "model_checkpoint": null,
    "framework": null,
    "model_family": "TinyTimeMixer",
    "modality": "timeseries",
    "provenance": "pretrained",
    "base_model_id": null,
    "usage_modes": [],
    "output_type": null,
    "context_length": 512,
    "prediction_length": 96,
    "domain": "finance",
    "frequency": null,
    "source": null,
    "artifact_path": null,
    "hf_repo": "ibm-granite/granite-timeseries-ttm-r1",
    "remote_endpoint": null,
    "pipeline_type": null,
    "sktime_class": "sktime.forecasting.ttm.TinyTimeMixerForecaster",
    "params": {
      "model_path": "ibm-granite/granite-timeseries-ttm-r1"
    },
    "param_hints": {},
    "training_regime": null,
    "trained_on": null,
    "tags": [
      "foundation",
      "tiny"
    ],
    "status": "active",
    "version"

## 14. `register_finetuned` - a fine-tune, with lineage

Say you fine-tuned TTM-R1 on Chiller 6 telemetry. This points a card at the checkpoint.

**Expect:** `sktime_class` **inherited** from the base, `params.model_path` = your checkpoint,
`provenance = finetuned`, `base_model_id = hub_ttm_r1`.

In [10]:
r = tu.run({
    'name': 'tsfm.register_finetuned',
    'arguments': {
        'model_id': 'hub_ttm_ft',
        'checkpoint_path': '/artifacts/hub_ttm_ft',
        'base_model_id': 'hub_ttm_r1',
        'context_length': 512,
        'prediction_length': 96,
        'description': 'TTM-R1 fine-tuned on Chiller 6 telemetry',
        'domain': 'energy',
    },
})
print(json.dumps(r, indent=2, default=str))

{
  "result": {
    "model_id": "hub_ttm_ft",
    "description": "TTM-R1 fine-tuned on Chiller 6 telemetry",
    "task_ids": [
      "tsfm_forecasting"
    ],
    "model_checkpoint": "/artifacts/hub_ttm_ft",
    "framework": null,
    "model_family": null,
    "modality": "timeseries",
    "provenance": "finetuned",
    "base_model_id": "hub_ttm_r1",
    "usage_modes": [],
    "output_type": null,
    "context_length": 512,
    "prediction_length": 96,
    "domain": "energy",
    "frequency": null,
    "source": "local_artifact",
    "artifact_path": "/artifacts/hub_ttm_ft",
    "hf_repo": null,
    "remote_endpoint": null,
    "pipeline_type": null,
    "sktime_class": "sktime.forecasting.ttm.TinyTimeMixerForecaster",
    "params": {
      "model_path": "/artifacts/hub_ttm_ft"
    },
    "param_hints": {},
    "training_regime": null,
    "trained_on": null,
    "tags": [],
    "status": "active",
    "version": "1",
    "created_by": "agent.tsfm.finetune",
    "created_at": "2026-07-

### A known bug: an unknown base is accepted

Here the `base_model_id` does not exist. Watch what happens.

**Expect (this is the bug):** the card is **accepted**, and `sktime_class` silently falls back
to `TinyTimeMixerForecaster` - which may be the wrong architecture entirely. The tool
"succeeds" and the output looks plausible, which is what makes it dangerous. It is documented
in the tool's docstring as a caveat, and not yet fixed.

In [14]:
r = tu.run({
    'name': 'tsfm.register_finetuned',
    'arguments': {
        'model_id': 'hub_orphan_ft',
        'checkpoint_path': '/artifacts/orphan',
        'base_model_id': 'does_not_exist',
        'context_length': 96,
        'prediction_length': 28,
        'description': 'fine-tune whose base does not exist',
    },
})
card = r.get('result', r)
print('base_model_id:', card.get('base_model_id'))
print('sktime_class :', card.get('sktime_class'), ' <-- silently defaulted')

base_model_id: None
sktime_class : None  <-- silently defaulted


## 17. `new_model_version` - supersede r1 with r2

HF's own R1 card declares `new_version: granite-timeseries-ttm-r2`, so let's follow it.

**Expect:** a new card `hub_ttm_r2` with `version = 2` and `supersedes = hub_ttm_r1`.
The predecessor flips to `status = superseded`.

In [15]:
r = tu.run({
    'name': 'tsfm.new_model_version',
    'arguments': {
        'model_id': 'hub_ttm_r1',
        'fields': {'hf_repo': R2, 'params': {'model_path': R2}},
        'new_model_id': 'hub_ttm_r2',
    },
})
print(json.dumps(r, indent=2, default=str))

{
  "result": {
    "model_id": "hub_ttm_r2",
    "description": "IBM Granite TinyTimeMixer R1: 805K-param tiny TS foundation forecaster.",
    "task_ids": [
      "tsfm_forecasting"
    ],
    "model_checkpoint": null,
    "framework": null,
    "model_family": "TinyTimeMixer",
    "modality": "timeseries",
    "provenance": "pretrained",
    "base_model_id": null,
    "usage_modes": [],
    "output_type": null,
    "context_length": 512,
    "prediction_length": 96,
    "domain": "finance",
    "frequency": null,
    "source": null,
    "artifact_path": null,
    "hf_repo": "ibm-granite/granite-timeseries-ttm-r2",
    "remote_endpoint": null,
    "pipeline_type": null,
    "sktime_class": "sktime.forecasting.ttm.TinyTimeMixerForecaster",
    "params": {
      "model_path": "ibm-granite/granite-timeseries-ttm-r2"
    },
    "param_hints": {},
    "training_regime": null,
    "trained_on": null,
    "tags": [
      "foundation",
      "tiny"
    ],
    "status": "active",
    "version"

## 16. `deprecate_model` - retire r1

A **soft delete**: the document stays in CouchDB, it just drops out of active listings.
Reversible with `update_model(model_id, {'status': 'active'})`.

**Expect:** `status = deprecated` plus your `deprecation_reason`.

In [16]:
r = tu.run({
    'name': 'tsfm.deprecate_model',
    'arguments': {'model_id': 'hub_ttm_r1', 'reason': 'superseded by R2 (per the HF card)'},
})
print(json.dumps(r, indent=2, default=str))

{
  "result": {
    "model_id": "hub_ttm_r1",
    "description": "IBM Granite TinyTimeMixer R1: 805K-param tiny TS foundation forecaster.",
    "task_ids": [
      "tsfm_forecasting"
    ],
    "model_checkpoint": null,
    "framework": null,
    "model_family": "TinyTimeMixer",
    "modality": "timeseries",
    "provenance": "pretrained",
    "base_model_id": null,
    "usage_modes": [],
    "output_type": null,
    "context_length": 512,
    "prediction_length": 96,
    "domain": "finance",
    "frequency": null,
    "source": null,
    "artifact_path": null,
    "hf_repo": "ibm-granite/granite-timeseries-ttm-r1",
    "remote_endpoint": null,
    "pipeline_type": null,
    "sktime_class": "sktime.forecasting.ttm.TinyTimeMixerForecaster",
    "params": {
      "model_path": "ibm-granite/granite-timeseries-ttm-r1"
    },
    "param_hints": {},
    "training_regime": null,
    "trained_on": null,
    "tags": [
      "foundation",
      "tiny"
    ],
    "status": "deprecated",
    "vers

## What is left in the catalog?

**Expect:** `hub_ttm_r1` is **gone from the active list** (deprecated), while `hub_ttm_r2`,
the fine-tunes, and the seeded `ttm_96_28` remain.

In [17]:
r = tu.run({'name': 'tsfm.list_models', 'arguments': {}})
models = r.get('result', r).get('models', [])
for m in sorted(models, key=lambda x: x['model_id']):
    print(f"  {m['model_id']:<18} {m.get('status'):<10} {m.get('provenance')}")

print('\nhub_ttm_r1 in the active list?', any(m['model_id'] == 'hub_ttm_r1' for m in models))

  hub_orphan_ft      active     finetuned
  hub_ttm_ft         active     finetuned
  hub_ttm_r2         active     pretrained
  ttm_96_28          active     pretrained
  ttm_r1_chiller6    active     finetuned
  ttm_r2_512_96      active     pretrained

hub_ttm_r1 in the active list? False


### The deprecated card is still in the database

It disappeared from `list_models` but the document is still there - that is what "soft delete"
means. You can see it in Fauxton too.

In [18]:
try:
    doc = couch('/model_catalog/model:hub_ttm_r1')
    print('still in CouchDB ->', doc['_id'], '| status:', doc['status'])
    print('deprecation_reason:', doc.get('deprecation_reason'))
except Exception as e:
    print('(memory store, or not reachable):', e)

still in CouchDB -> model:hub_ttm_r1 | status: deprecated
deprecation_reason: superseded by R2 (per the HF card)


## Clean up

Close the stdio session. To reset the catalog, re-run `init_data.py --reset`.

Note the notebook is **not idempotent**: `register_model` overwrites by `model_id`, and
`new_model_version` bumps the version each run, so ids drift on repeat runs. Reset between runs.

In [19]:
tu.close()
print('closed')

closed
